## 04 - Impact of Room Type on Price

- **Research Question:** How are accommodation type (`room_type`) and listing prices associated? How much cheaper is a private room compared to an entire apartment (the "privacy discount"), and is this pattern consistent across different cities?
- **Variables:**
  - `room_type`
  - `price`
- **Statistical Tools:**
  - **Conditional Statistics:** Calculating the sample mean and sample median for each specific room type in every city.
  - **The Privacy Discount:** Measuring how much cheaper private rooms are compared to entire homes.
  - **Hypothesis Testing:**
    - **One-way ANOVA (F-test):** Checking if average prices differ significantly across all room categories.
    - **Welch's t-test and Mann-Whitney U test:** Comparing prices directly between `Entire home/apt` and `Private room` using both parametric and non-parametric tests to handle skewed data.
  - **Visualizations:**
    - Multi-panel side-by-side boxplots showing price spreads and medians across room types for each city.
    - Comparative charts illustrating how privacy discounts vary across global locations.

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from src.cities import CITIES
from src.load import load_all
from src.study_utils import export_table

cols = ['id', 'price', 'room_type', 'latitude', 'longitude', 'neighbourhood_cleansed']
data = load_all(usecols=cols)

data = data.dropna(subset=['price']).copy()

data['city_name'] = data['city'].map({k: v.name for k, v in CITIES.items()})
data['currency'] = data['city'].map({k: v.currency for k, v in CITIES.items()})

# Illustrative FX assumptions, not sourced historical exchange rates.
fx_to_eur = {
    'EUR': 1.00,
    'USD': 0.86,
    'JPY': 0.0055,
    'MXN': 0.05
}
data['fx_to_eur'] = data['currency'].map(fx_to_eur)
data['price_eur'] = data['price'] * data['fx_to_eur']

num_offers = len(data)
num_cities = data['city'].nunique()
print(f"Loaded {num_offers:,} listings with valid price and room_type across {num_cities} cities.")

Loaded 154,226 listings with valid price and room_type across 6 cities.


In [2]:
records = []
room_order = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']

for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    total_city = len(city_df)
    
    for r_type in room_order:
        sub = city_df[city_df['room_type'] == r_type]['price']
        if len(sub) > 0:
            records.append({
                'City': city_obj.name,
                'Currency': city_obj.currency,
                'Room Type': r_type,
                'Listings Count': len(sub),
                'Market Share (%)': round((len(sub) / total_city) * 100, 1),
                'Median Price': round(sub.median(), 2),
                'Mean Price': round(sub.mean(), 2),
                'Std Dev (s)': round(sub.std(), 2),
                'IQR': round(sub.quantile(0.75) - sub.quantile(0.25), 2)
            })

#Conditional Price Statistics by Room Type
df_room_stats = pd.DataFrame(records)
df_room_pivot = df_room_stats.pivot(index=['City', 'Currency'], columns='Room Type', values=['Listings Count', 'Median Price', 'Mean Price'])
df_room_pivot

Listings Count                                      \
Room Type            Entire home/apt Hotel room Private room Shared room   
City        Currency                                                       
Barcelona   EUR              10114.0       62.0       3082.0        97.0   
Los Angeles USD              28437.0      400.0       9069.0       280.0   
Madrid      EUR              13936.0       29.0       5065.0       142.0   
Mexico City MXN              19856.0       83.0       9407.0       291.0   
New York    USD              11537.0      480.0       9306.0       192.0   
Tokyo       JPY              27150.0      112.0       4787.0       312.0   

                        Median Price                                      \
Room Type            Entire home/apt Hotel room Private room Shared room   
City        Currency                                                       
Barcelona   EUR               221.50     196.10        64.00       46.00   
Los Angeles USD               282.00     319.19        85.37       36.82   
Madrid      EUR               144.00     104.00        64.00       25.00   
Mexico City MXN              2163.00    3132.00       970.00      503.00   
New York    USD               222.87     447.00       102.96       54.66   
Tokyo       JPY             20537.16   11155.34     15000.00     3424.00   

                          Mean Price                                      
Room Type            Entire home/apt Hotel room Private room Shared room  
City        Currency                                                      
Barcelona   EUR               284.77     252.49       100.74       73.97  
Los Angeles USD               463.59     511.52       135.77       76.25  
Madrid      EUR               202.80     107.70       121.21       62.00  
Mexico City MXN              3472.76    7594.79      1758.79      765.40  
New York    USD               331.41     721.57       191.78      164.39  
Tokyo       JPY             32124.00   31567.57     26838.11     5613.19

In [3]:
# Statistical Hypothesis Testing: ANOVA, Welch's t-test, and Mann-Whitney U test
test_results = []

for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    
    # 1. One-way ANOVA across all room types
    groups = [group['price'].values for _, group in city_df.groupby('room_type')]
    f_stat, p_anova = stats.f_oneway(*groups)
    
    # 2. Entire home vs Private room comparisons
    entire = city_df[city_df['room_type'] == 'Entire home/apt']['price']
    private = city_df[city_df['room_type'] == 'Private room']['price']
    
    med_entire = entire.median()
    med_private = private.median()
    discount_pct = (1 - med_private / med_entire) * 100
    
    # Parametric Welch's t-test (unequal variances)
    t_stat, p_ttest = stats.ttest_ind(entire, private, equal_var=False)
    
    # Non-parametric Mann-Whitney U test
    u_stat, p_mwu = stats.mannwhitneyu(entire, private, alternative='two-sided')
    
    test_results.append({
        'City': city_obj.name,
        'Currency': city_obj.currency,
        'Median Entire': round(med_entire, 2),
        'Median Private': round(med_private, 2),
        'Private Discount (%)': round(discount_pct, 1),
        'ANOVA F-stat': round(f_stat, 1),
        'ANOVA p-val': '< 0.0001' if p_anova < 0.0001 else f'{p_anova:.4f}',
        'Welch t-stat': round(t_stat, 1),
        'Welch p-val': '< 0.0001' if p_ttest < 0.0001 else f'{p_ttest:.4f}',
        'Mann-Whitney U': f'{u_stat:,.0f}',
        'MWU p-val': '< 0.0001' if p_mwu < 0.0001 else f'{p_mwu:.4f}'
    })

# Statistical Significance of Price Differences by Room Type
df_tests = pd.DataFrame(test_results).set_index('City')
df_tests

,Currency,Median Entire,Median Private,Private Discount (%),ANOVA F-stat,ANOVA p-val,Welch t-stat,Welch p-val,Mann-Whitney U,MWU p-val
City,,,,,,,,,,
Barcelona,EUR,221.50,64.00,71.1,252.8,< 0.0001,42.7,< 0.0001,"26,280,698",< 0.0001
Los Angeles,USD,282.00,85.37,69.7,611.9,< 0.0001,65.0,< 0.0001,"222,968,564",< 0.0001
Madrid,EUR,144.00,64.00,55.6,29.2,< 0.0001,8.9,< 0.0001,"59,100,942",< 0.0001
New York,USD,222.87,102.96,53.8,242.5,< 0.0001,20.5,< 0.0001,"80,511,214",< 0.0001
Tokyo,JPY,20537.16,15000.00,27.0,24.7,< 0.0001,5.3,< 0.0001,"82,335,430",< 0.0001
Mexico City,MXN,2163.00,970.00,55.2,83.4,< 0.0001,12.4,< 0.0001,"148,666,739",< 0.0001


In [4]:
# Side-by-Side Boxplots by Room Type Across All 6 Cities
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

colors_room = {
    'Entire home/apt': '#1d3557',
    'Private room': '#457b9d',
    'Hotel room': '#f4a261',
    'Shared room': '#e63946'
}

for idx, (city_key, city_obj) in enumerate(CITIES.items()):
    ax = axes[idx]
    city_df = data[data['city'] == city_key]
    
    # Calculate boxes and medians from the same full sample; hide only outlier points.
    p98 = city_df['price'].quantile(0.98)
    city_vis = city_df
    
    sns.boxplot(
        data=city_vis, x='room_type', y='price', order=room_order,
        hue='room_type', legend=False,
        palette=colors_room, ax=ax, width=0.55, showfliers=False
    )
    
    # Annotate medians
    medians = city_df.groupby('room_type')['price'].median().reindex(room_order)
    for r_idx, (r_type, med) in enumerate(medians.items()):
        if pd.notna(med):
            ax.text(r_idx, med + p98 * 0.02, f'{med:,.0f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
            
    groups = [g['price'].values for _, g in city_df.groupby('room_type')]
    f_stat, p_val = stats.f_oneway(*groups)
    
    ax.set_title(f"{city_obj.name} ({city_obj.currency})\nANOVA: F = {f_stat:.1f} (p = {p_val:.2e})", fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel(f"Price ({city_obj.currency})", fontsize=10)
    ax.set_xlabel('')
    ax.set_xticks(range(len(room_order)))
    ax.set_xticklabels(['Entire', 'Private', 'Hotel', 'Shared'], fontsize=9, fontweight='bold')

plt.suptitle("Figure 1: Price Boxplots by Room Type Across All 6 Cities\nDrastic Price Drop from Entire Homes to Private & Shared Rooms", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study4_room_type_boxplots_by_city.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11843/3685477681.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# The Privacy Premium and Cross-City Room Type Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

discount_data = []
for city_key, city_obj in CITIES.items():
    city_df = data[data['city'] == city_key]
    med_entire = city_df[city_df['room_type'] == 'Entire home/apt']['price'].median()
    med_priv = city_df[city_df['room_type'] == 'Private room']['price'].median()
    discount = (1 - med_priv / med_entire) * 100
    discount_data.append({'City': city_obj.name, 'Discount': discount})

df_disc = pd.DataFrame(discount_data).sort_values('Discount', ascending=False)
bars = ax1.bar(df_disc['City'], df_disc['Discount'], color='#1d3557', width=0.55)
ax1.set_title('Private Room Discount vs Entire Home (Median %)', fontsize=12, fontweight='bold', pad=10)
ax1.set_ylabel('Discount (%) [1 - Median(Private) / Median(Entire)]', fontsize=10)
ax1.set_ylim(0, 85)
ax1.tick_params(axis='x', rotation=15)

for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1.5, f"{h:.1f}%", ha='center', va='bottom', fontsize=9.5, fontweight='bold')

city_order = ['Barcelona', 'Los Angeles', 'Madrid', 'New York', 'Mexico City', 'Tokyo']
sub_data = data[data['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

sns.boxplot(
    data=sub_data, x='city_name', y='price_eur', hue='room_type', order=city_order,
    palette={'Entire home/apt': '#1d3557', 'Private room': '#457b9d'}, ax=ax2,
    showfliers=False, width=0.65
)
ax2.set_ylim(0, 450)
ax2.set_title('Price Comparison: Entire Home vs Private Room (Illustrative FX assumptions)', fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('Price (EUR equivalent)', fontsize=10)
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=15)
ax2.legend(title='Room Type', loc='upper right', frameon=True)

plt.suptitle("Figure 2: The Privacy Premium in Airbnb Pricing Across Global Cities", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/study4_privacy_premium_and_discount.png', dpi=300, bbox_inches='tight')
plt.show()

/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11843/3872221526.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Findings
Private-room medians are below entire-home medians in every city: relative gaps are 71.1% in Barcelona, 69.7% in Los Angeles, 55.6% in Madrid, 53.8% in New York, 27.0% in Tokyo and 55.2% in Mexico City. These compare observed room-type groups and do not hold capacity, location or date constant. Room-type shares in this notebook use the **priced sample** as their denominator.

ANOVA, Welch and Mann–Whitney results are computed above. ANOVA relies on equal variances, which are questionable for these skewed samples. Welch relaxes equal variance for the two-group mean comparison; Mann–Whitney compares distributions and is not automatically a pure median test. Large samples and shared hosts require caution when interpreting p-values. No measured floor area or tourist preferences support a causal explanation for Tokyo's smaller gap. FX figures use the same explicitly assumed rates as study 03.


In [6]:
import plotly.express as px

fig = px.scatter_geo(
    data.dropna(subset=["latitude", "longitude"]),
    lat="latitude",
    lon="longitude",
    color="room_type",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hover_data={"room_type": True, "price_eur": ":.1f", "neighbourhood_cleansed": True, "latitude": False, "longitude": False},
    projection="natural earth",
    opacity=0.5,
    size_max=6,
    title="Spatial Distribution of Room Types Across 6 Cities (Study 04)",
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.write_html('../maps/study04_roomtypes_global.html', include_plotlyjs=True)
print('Saved maps/study04_roomtypes_global.html')

Saved maps/study04_roomtypes_global.html


### Spatial interpretation
The categorical point maps locate observed room types. Overlapping markers and scale can hide listings. Apparent clusters are descriptive and do not establish segregation, neighbourhood effects or a spatial explanation of the price gap.


In [7]:
import plotly.express as px

CITY = "Tokyo"
df = data[data["city_name"] == CITY].copy()

fig = px.scatter_map(
    df,
    lat="latitude",
    lon="longitude",
    color="room_type",
    hover_data=["room_type", "price", "neighbourhood_cleansed"],
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"Category Map: Room Types in {CITY}",
    zoom=12,
    center={"lat": df["latitude"].median(), "lon": df["longitude"].median()},
    opacity=0.6,
    map_style="carto-positron",
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.write_html('../maps/study04_roomtypes_tokyo.html', include_plotlyjs=True)
print('Saved maps/study04_roomtypes_tokyo.html')


Saved maps/study04_roomtypes_tokyo.html


In [8]:
export_table(df_room_stats, 'study04_room_stats')
export_table(df_tests, 'study04_tests', index=True)
export_table(df_disc, 'study04_discounts')


,City,Discount
0,Barcelona,71.106095
1,Los Angeles,69.726950
2,Madrid,55.555556
5,Mexico City,55.154877
3,New York,53.800422
4,Tokyo,26.961681
